# Ilera-ATLAS · 01_extract — Standing Orders → clean chunks

**Runbook Part 2 · Fri 2 Oct.** Downloads the three 2024 National Standing Orders (CHEW, JCHEW, CHO),
removes running headers and table-of-contents pages, splits the text by numbered section
(e.g. `2.27 WORM INFESTATION`), cuts long sections into ~900-word chunks, and uploads everything
to your private Hugging Face dataset `ilera-data/raw/`.

**Kaggle settings:** Accelerator **None** (no GPU needed, saves quota) · Internet **On** ·
Add-ons → Secrets → `HF_TOKEN` attached. Also runs on Colab (secret `HF_TOKEN`) or locally (env var `HF_TOKEN`).

Edit only the **Config** cell. Then *Run All*. Takes about 2–4 minutes.

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"        # <- change this
DATA_REPO = f"{HF_USER}/ilera-data"  # private dataset repo (created if missing)
MAX_WORDS = 900                      # chunk size; N-ATLaS context is ~8k tokens
DRAFT_DOCS = ["CHEW"]                # docs Part 3 drafts Q&A from (JCHEW/CHO kept for reference)
PROBE_SECTIONS = []                  # e.g. ["2.19", "2.21", "2.29"]; chosen by the clinical reviewer (cell 6)
UPLOAD = True                        # False = dry run, nothing sent to Hugging Face

In [ ]:
import os, subprocess, sys
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)   # deprecated Kaggle default -> removes the FutureWarning
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"         # its replacement: faster model downloads
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf", "huggingface_hub", "pandas"], capture_output=True, text=True)
if r.returncode != 0:                               # pip prints harmless dependency warnings; show them only on failure
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install failed")
print("packages installed")

In [ ]:
import os

def get_hf_token():
    try:                                            # Kaggle
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:                                            # Colab
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")               # local

HF_TOKEN = get_hf_token()
if UPLOAD:
    assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
    assert HF_TOKEN, "No HF_TOKEN found: add it as a Kaggle/Colab secret or env var"
    from huggingface_hub import whoami
    print("Logged in to Hugging Face as", whoami(token=HF_TOKEN)["name"])

## 1. Download the PDFs
The PHC Workforce Hub server rejects requests without browser-like headers (HTTP 406), so we send them.
Each file is checked to really be a PDF and its SHA-256 is recorded in `sources.json` for the model card.

In [ ]:
import hashlib, json, requests

BASE = "https://phcworkforcehub.org.ng/wp-content/uploads/2024/10/"
PDFS = {
    "CHEW":  BASE + "CHEW-NATIONAL-STANDING-ORDERS-Section-1-7-Protrait-CHEW.pdf",
    "JCHEW": BASE + "JCHEW-Text-only-1-7-Sections.pdf",
    "CHO":   BASE + "CHO-NATIONAL-STANDING-ORDERS-Section-1-8-Protrait-UPDATED.pdf",
}
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/pdf,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}
os.makedirs("pdfs", exist_ok=True)
sources = []
for name, url in PDFS.items():
    path = f"pdfs/{name}.pdf"
    if not os.path.exists(path):
        r = requests.get(url, headers=HEADERS, timeout=600)
        r.raise_for_status()
        if not r.content.startswith(b"%PDF"):
            raise RuntimeError(f"{name}: server did not return a PDF "
                               f"({r.headers.get('content-type')}). Download it in a browser "
                               f"and upload it to pdfs/{name}.pdf")
        open(path, "wb").write(r.content)
    data = open(path, "rb").read()
    sources.append({"doc": name, "url": url, "bytes": len(data),
                    "sha256": hashlib.sha256(data).hexdigest(),
                    "publisher": "Community Health Practitioners Registration Board of Nigeria "
                                 "(CHPRBN) with FMoH and NPHCDA, revised 2024"})
    print(f"{name}: {len(data) / 1e6:.1f} MB  sha256 {sources[-1]['sha256'][:16]}")

## 2. Extraction functions
* **Running headers** (a line on more than half of the pages, e.g. the document title) and `PAGE n` lines are removed.
* **Soft hyphens** from the PDF (`Pregnan-cy`) are joined back.
* **Table-of-contents pages** are skipped: a page with more than 5 section headings is a contents page.
* **Section headings** look like `2.27 WORM INFESTATION` or `1.3.6 CONDITIONS AFFECTING THE EYES`.
* Long sections are split into ~`MAX_WORDS` chunks; a short tail (<150 words) is merged into the previous chunk.

Known limit: the three-column tables (findings / clinical judgement / action) come out column by
column, not row by row. Text is complete but sometimes out of order; reviewers fix this at the Q&A
stage (Part 4). Do not spend more time on perfect table extraction.

In [ ]:
import collections, re
import pymupdf

HEADING = re.compile(r"^(\d{1,2}(?:\.\d{1,2}){1,2})\.?\s+([A-Z][A-Z0-9 ,/&’'()\-]{2,80})$")
PAGE_NO = re.compile(r"^PAGE\s+\d+$")


def page_lines(doc):
    """Yield (page_no, [clean lines]) with running headers and page numbers removed."""
    raw = [[l.strip() for l in p.get_text().replace("\u00ad\n", "").replace("\u00ad", "").splitlines()]
           for p in doc]
    counts = collections.Counter(l for lines in raw for l in set(lines) if l)
    running = {l for l, n in counts.items() if n > 0.5 * len(raw) and l.isupper() and len(l) > 20}
    for i, lines in enumerate(raw):
        yield i + 1, [l for l in lines if l and l not in running and not PAGE_NO.match(l)]


def sections(path):
    doc = pymupdf.open(path)
    cur = {"section_id": "0.0", "title": "FRONT MATTER", "start": 1, "pages": [], "lines": []}
    out, pages_out = [], []
    for pno, lines in page_lines(doc):
        pages_out.append({"page": pno, "text": "\n".join(lines)})
        heads = [i for i, l in enumerate(lines) if HEADING.match(l)]
        if len(heads) > 5:                         # table-of-contents page
            continue
        prev = 0
        for i in heads:
            cur["lines"] += lines[prev:i]
            if cur["lines"]:
                cur["pages"].append(pno)
                out.append(cur)
            m = HEADING.match(lines[i])
            cur = {"section_id": m.group(1), "title": m.group(2).strip(),
                   "start": pno, "pages": [], "lines": []}
            prev = i + 1
        cur["lines"] += lines[prev:]
        cur["pages"].append(pno)
    out.append(cur)
    secs = [s for s in out if s["section_id"] != "0.0" and len(" ".join(s["lines"]).split()) > 30]
    return secs, pages_out


def chunk_section(name, s):
    parts, buf, words = [], [], 0
    for l in s["lines"]:
        buf.append(l); words += len(l.split())
        if words >= MAX_WORDS:
            parts.append(buf); buf, words = [], 0
    if buf:
        if parts and words < 150:
            parts[-1] += buf
        else:
            parts.append(buf)
    pages = sorted(set(s["pages"])) or [s["start"]]
    page_str = f"{pages[0]}-{pages[-1]}" if pages[0] != pages[-1] else str(pages[0])
    return [{
        "chunk_id": f"{name}-{s['section_id']}-{k + 1}",
        "doc": name, "section_id": s["section_id"], "title": s["title"],
        "pages": page_str, "part": f"{k + 1}/{len(parts)}",
        "source": f"{name} Standing Orders 2024, Section {s['section_id']} {s['title'].title()}, p. {page_str}",
        "draft": name in DRAFT_DOCS,
        "text": "\n".join(part),
    } for k, part in enumerate(parts)]

## 3. Run extraction

In [ ]:
chunks, pages = [], []
for name in PDFS:
    secs, pgs = sections(f"pdfs/{name}.pdf")
    pages += [{"doc": name, **p} for p in pgs]
    for s in secs:
        chunks += chunk_section(name, s)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl("pages.jsonl", pages)
write_jsonl("chunks.jsonl", chunks)
json.dump(sources, open("sources.json", "w"), indent=2)
print(len(pages), "pages ->", len(chunks), "chunks")

## 4. Quality report and self-checks
If an `assert` fails, the PDF layout probably changed: stop and look at the printed sample before continuing.

In [ ]:
import pandas as pd
df = pd.DataFrame(chunks)
df["words"] = df.text.str.split().str.len()
report = df.groupby("doc").agg(sections=("section_id", "nunique"), chunks=("chunk_id", "count"),
                               median_words=("words", "median"), max_words=("words", "max"),
                               total_words=("words", "sum"))
print(report.to_string())

assert df.chunk_id.is_unique, "duplicate chunk ids"
assert (report.sections >= 100).all(), "too few sections found: heading pattern no longer matches"
assert df.words.max() <= MAX_WORDS + 400, "a chunk is far too long"
assert not df.text.str.contains("NATIONAL STANDING ORDERS FOR", regex=False).any(), "running header leaked"
for name in PDFS:
    assert (df[df.doc == name].section_id == "2.3").any(), f"{name}: section 2.3 (FEVER) missing"

print(f"\nDrafting set ({', '.join(DRAFT_DOCS)}): {int(df.draft.sum())} chunks")
sample = df[(df.doc == "CHEW") & (df.title.str.startswith("WORM"))].iloc[0]
print("\nSample:", sample.chunk_id, "|", sample.source)
print(sample.text[:700])

## 5. Section index for the team
`sections_index.csv` lists every section with its pages and size. Share it with the clinical
reviewer: it is the menu for picking the probe sections (next cell) and for splitting review work.

In [ ]:
index = (df.groupby(["doc", "section_id", "title"], sort=False)
           .agg(pages=("pages", "first"), chunks=("chunk_id", "count"), words=("words", "sum"))
           .reset_index())
index.to_csv("sections_index.csv", index=False, encoding="utf-8-sig")
print(index[index.doc == "CHEW"].to_string(index=False, max_rows=200))

## 6. Probe sections (unseen-section test)
The clinical reviewer picks **3 to 4 sections** that will get **no training data at all** (see the
runbook correction note). Choose ordinary, self-contained topics (e.g. an eye, skin or ear/nose/throat
section), not core topics like fever, malaria or pregnancy. Put their section numbers in
`PROBE_SECTIONS` in the Config cell and re-run from here.

A probe section is excluded in **all three** documents, so the topic cannot leak in through JCHEW or CHO. The same topic often appears twice (a child section and an adult section, e.g. EYE DISORDERS 2.19 and 3.18): include **both**. The cell below warns you about look-alike sections.

In [ ]:
if not PROBE_SECTIONS:
    print("PROBE_SECTIONS is empty: pick 3-4 sections from the index above, set them in Config, "
          "and re-run cells 6-7. Uploading without a probe file for now.")
probe_ids = df[df.section_id.isin(PROBE_SECTIONS)].chunk_id.tolist()
unknown = set(PROBE_SECTIONS) - set(df.section_id)
assert not unknown, f"not a section number: {unknown}"
open("probe_chunks.txt", "w").write("\n".join(probe_ids))
if PROBE_SECTIONS:
    print(len(probe_ids), "probe chunks:")
    print(index[index.section_id.isin(PROBE_SECTIONS)].to_string(index=False))
    # the same topic often has a child section and an adult section (e.g. EYE DISORDERS 2.19 and 3.18)
    words = {w for t in index[index.section_id.isin(PROBE_SECTIONS)].title for w in t.split() if len(w) >= 3}
    words -= {"DISORDERS", "CONDITIONS", "AFFECTING", "PROBLEMS", "HEALTH", "WITH", "THE", "AND", "FOR", "CARE"}
    similar = index[~index.section_id.isin(PROBE_SECTIONS) & index.title.apply(lambda t: bool(words & set(t.split())))]
    if len(similar):
        print("\nWARNING: these other sections look like the same topic. Add them to PROBE_SECTIONS "
              "unless the reviewer is sure they are different, otherwise the probe topic leaks into training:")
        print(similar.drop_duplicates("section_id")[["section_id", "title", "pages"]].to_string(index=False))

## 7. Upload to Hugging Face (`ilera-data/raw/`)

In [ ]:
FILES = ["chunks.jsonl", "pages.jsonl", "sections_index.csv", "probe_chunks.txt", "sources.json"]
if UPLOAD:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(DATA_REPO, repo_type="dataset", private=True, exist_ok=True)
    for f in FILES:
        api.upload_file(path_or_fileobj=f, path_in_repo=f"raw/{f}", repo_id=DATA_REPO,
                        repo_type="dataset", commit_message=f"01_extract: {f}")
    print("Uploaded:", [f"raw/{f}" for f in FILES])
    print(f"https://huggingface.co/datasets/{DATA_REPO}/tree/main/raw")
else:
    print("Dry run: files written locally only:", FILES)

## Done when
* `raw/chunks.jsonl` and `raw/sections_index.csv` are on Hugging Face.
* 5 random chunks read sensibly (open `chunks.jsonl` in the HF viewer).
* `raw/probe_chunks.txt` holds the reviewer's 3–4 probe sections (can be added later today).

**Next:** Part 3 (`02_draft`), Sat 3 Oct. It reads `raw/chunks.jsonl` and `raw/probe_chunks.txt`
and drafts only chunks with `"draft": true`.